# Create Data Subset: SSP245 Climate Variables (2020-2050)

This notebook creates a subset of the H3 DGGS climate data containing only:
- **Variables**: `ssp245_tx_max` and `ssp245_prcptot` (with all percentiles)
- **Time range**: 2020-2050

## Overview

1. Load the full hierarchical Zarr store
2. Select variables matching the pattern
3. Filter to the time range 2020-2050
4. Save as a new Zarr store


## Setup


In [1]:
import xarray as xr
import zarr
import pandas as pd
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

print("✅ Packages imported")


✅ Packages imported


## Configuration


In [12]:
# Input: Full hierarchical Zarr store
INPUT_ZARR = Path("outputs/dggs/h3/canada_climate_h3_dggs.zarr")

SSP = "ssp370"

# Variables to include (will match all percentiles)
VARIABLE_PATTERNS = [f"{SSP}_tx_max", f"{SSP}_prcptot"]

# Time range
START_YEAR = 2020
END_YEAR = 2030

# Output: Subset Zarr store
OUTPUT_ZARR = Path(f"outputs/dggs/h3/canada_climate_h3_{SSP}_{START_YEAR}-{END_YEAR}.zarr")

print(f"Configuration:")
print(f"  Input:  {INPUT_ZARR}")
print(f"  Output: {OUTPUT_ZARR}")
print(f"  Variables: {VARIABLE_PATTERNS}")
print(f"  Time range: {START_YEAR}-{END_YEAR}")


Configuration:
  Input:  outputs/dggs/h3/canada_climate_h3_dggs.zarr
  Output: outputs/dggs/h3/canada_climate_h3_ssp370_2020-2030.zarr
  Variables: ['ssp370_tx_max', 'ssp370_prcptot']
  Time range: 2020-2030


## Explore Input Data


In [13]:
# Check what's in the input
zarr_root = zarr.open_group(INPUT_ZARR, mode='r')
groups = sorted([g for g in zarr_root.group_keys()])

print(f"\nInput contains {len(groups)} resolution groups: {groups}\n")

# Check a sample group to see available variables
sample_ds = xr.open_zarr(INPUT_ZARR, group=groups[0], decode_timedelta=False)
all_vars = sorted(sample_ds.data_vars)

print(f"Sample group '{groups[0]}' has {len(all_vars)} variables")
print(f"\nVariables matching our patterns:")

selected_vars = []
for pattern in VARIABLE_PATTERNS:
    matching = [v for v in all_vars if pattern in v]
    selected_vars.extend(matching)
    print(f"  {pattern}: {len(matching)} variables")
    for v in matching:
        print(f"    - {v}")

selected_vars = sorted(list(set(selected_vars)))
print(f"\nTotal selected variables: {len(selected_vars)}")

sample_ds.close()



Input contains 7 resolution groups: ['res0', 'res1', 'res2', 'res3', 'res4', 'res5', 'res6']

Sample group 'res0' has 24 variables

Variables matching our patterns:
  ssp370_tx_max: 3 variables
    - ssp370_tx_max_p10
    - ssp370_tx_max_p50
    - ssp370_tx_max_p90
  ssp370_prcptot: 3 variables
    - ssp370_prcptot_p10
    - ssp370_prcptot_p50
    - ssp370_prcptot_p90

Total selected variables: 6


## Create Subset for Each Resolution

Process each resolution group, filtering variables and time range.


In [14]:
from tqdm import tqdm

print(f"\n{'='*70}")
print(f"Creating subset Zarr store")
print(f"{'='*70}\n")

# Remove output if it exists
if OUTPUT_ZARR.exists():
    import shutil
    print(f"Removing existing output: {OUTPUT_ZARR}")
    shutil.rmtree(OUTPUT_ZARR)

OUTPUT_ZARR.parent.mkdir(parents=True, exist_ok=True)

# Process each resolution group
for i, group_name in enumerate(tqdm(groups, desc="Processing groups"), 1):
    tqdm.write(f"\n[{i}/{len(groups)}] Processing {group_name}...")

    # Load full dataset
    ds = xr.open_zarr(INPUT_ZARR, group=group_name, decode_timedelta=False)

    # Select only the variables we want
    vars_in_group = [v for v in selected_vars if v in ds.data_vars]

    if not vars_in_group:
        tqdm.write(f"  ⚠️  No matching variables in {group_name}, skipping")
        ds.close()
        continue

    ds_subset = ds[vars_in_group]

    # Filter time range
    if 'time' in ds_subset.dims:
        time_values = pd.to_datetime(ds_subset.time.values)

        # Select time range
        mask = (time_values.year >= START_YEAR) & (time_values.year <= END_YEAR)
        ds_subset = ds_subset.isel(time=mask)

        time_range = pd.to_datetime(ds_subset.time.values)
        tqdm.write(f"  Variables: {len(vars_in_group)}")
        tqdm.write(f"  Time: {time_range[0]} to {time_range[-1]} ({len(time_range)} steps)")
        tqdm.write(f"  Cells: {ds_subset.sizes['h3_id']:,}")
    else:
        tqdm.write(f"  Variables: {len(vars_in_group)}")
        tqdm.write(f"  Cells: {ds_subset.sizes['h3_id']:,}")

    # Write to output Zarr
    mode = 'w' if i == 1 else 'a'
    ds_subset.to_zarr(OUTPUT_ZARR, group=group_name, mode=mode, consolidated=False)

    ds.close()
    tqdm.write(f"  ✅ Saved to {group_name}")

# Consolidate metadata
print(f"\nConsolidating metadata...")
zarr.consolidate_metadata(OUTPUT_ZARR)

print(f"\n{'='*70}")
print(f"✅ Subset created successfully!")
print(f"{'='*70}")
print(f"Output: {OUTPUT_ZARR}")



Creating subset Zarr store



Processing groups:   0%|          | 0/7 [00:00<?, ?it/s]


[1/7] Processing res0...
  Variables: 6
  Time: 2020-01-01 00:00:00 to 2030-12-01 00:00:00 (132 steps)
  Cells: 9


Processing groups:  14%|█▍        | 1/7 [00:00<00:05,  1.14it/s]

  ✅ Saved to res0

[2/7] Processing res1...
  Variables: 6
  Time: 2020-01-01 00:00:00 to 2030-12-01 00:00:00 (132 steps)
  Cells: 35


...

  ✅ Saved to res4

[6/7] Processing res5...
  Variables: 6
  Time: 2020-01-01 00:00:00 to 2030-12-01 00:00:00 (132 steps)
  Cells: 45,661


Processing groups:  86%|████████▌ | 6/7 [00:05<00:01,  1.07s/it]

  ✅ Saved to res5

[7/7] Processing res6...
  Variables: 6
  Time: 2020-01-01 00:00:00 to 2030-12-01 00:00:00 (132 steps)
  Cells: 207,466


Processing groups: 100%|██████████| 7/7 [00:09<00:00,  1.38s/it]


  ✅ Saved to res6

Consolidating metadata...

✅ Subset created successfully!
Output: outputs/dggs/h3/canada_climate_h3_ssp370_2020-2030.zarr


## Verify Subset

Check the created subset to ensure it contains the correct data.


In [15]:
print(f"\n{'='*70}")
print(f"Verifying subset")
print(f"{'='*70}\n")

# Open output and verify
zarr_out = zarr.open_group(OUTPUT_ZARR, mode='r')
out_groups = sorted([g for g in zarr_out.group_keys()])

print(f"Output contains {len(out_groups)} resolution groups: {out_groups}\n")

# Check each group
for group_name in out_groups:
    ds = xr.open_zarr(OUTPUT_ZARR, group=group_name, decode_timedelta=False)

    print(f"{group_name}:")
    print(f"  Variables: {len(ds.data_vars)}")
    print(f"    {sorted(ds.data_vars)}")

    if 'time' in ds.dims:
        time_vals = pd.to_datetime(ds.time.values)
        print(f"  Time: {time_vals[0].year} to {time_vals[-1].year} ({len(time_vals)} steps)")

    print(f"  Cells: {ds.sizes['h3_id']:,}")
    print()

    ds.close()



Verifying subset

Output contains 7 resolution groups: ['res0', 'res1', 'res2', 'res3', 'res4', 'res5', 'res6']

res0:
  Variables: 6
    ['ssp370_prcptot_p10', 'ssp370_prcptot_p50', 'ssp370_prcptot_p90', 'ssp370_tx_max_p10', 'ssp370_tx_max_p50', 'ssp370_tx_max_p90']
  Time: 2020 to 2030 (132 steps)
  Cells: 9

res1:
  Variables: 6
    ['ssp370_prcptot_p10', 'ssp370_prcptot_p50', 'ssp370_prcptot_p90', 'ssp370_tx_max_p10', 'ssp370_tx_max_p50', 'ssp370_tx_max_p90']
  Time: 2020 to 2030 (132 steps)
  Cells: 35

res2:
  Variables: 6
    ['ssp370_prcptot_p10', 'ssp370_prcptot_p50', 'ssp370_prcptot_p90', 'ssp370_tx_max_p10', 'ssp370_tx_max_p50', 'ssp370_tx_max_p90']
  Time: 2020 to 2030 (132 steps)
  Cells: 189

res3:
  Variables: 6
    ['ssp370_prcptot_p10', 'ssp370_prcptot_p50', 'ssp370_prcptot_p90', 'ssp370_tx_max_p10', 'ssp370_tx_max_p50', 'ssp370_tx_max_p90']
  Time: 2020 to 2030 (132 steps)
  Cells: 1,123

res4:
  Variables: 6
    ['ssp370_prcptot_p10', 'ssp370_prcptot_p50', 'ssp370_p

## Size Comparison

Compare the size of the original and subset stores.


In [16]:
def get_dir_size(path):
    """Get total size of directory in bytes."""
    total = 0
    for entry in path.rglob('*'):
        if entry.is_file():
            total += entry.stat().st_size
    return total

def format_size(bytes_size):
    """Format bytes as human-readable string."""
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if bytes_size < 1024.0:
            return f"{bytes_size:.2f} {unit}"
        bytes_size /= 1024.0
    return f"{bytes_size:.2f} PB"

print(f"\n{'='*70}")
print(f"Size Comparison")
print(f"{'='*70}\n")

input_size = get_dir_size(INPUT_ZARR)
output_size = get_dir_size(OUTPUT_ZARR)
reduction = (1 - output_size / input_size) * 100

print(f"Original dataset: {format_size(input_size)}")
print(f"Subset dataset:   {format_size(output_size)}")
print(f"Reduction:        {reduction:.1f}%")



Size Comparison

Original dataset: 33.76 GB
Subset dataset:   644.62 MB
Reduction:        98.1%
